In [1]:
import pandas as pd
import numpy as np
import re
import os
import glob
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [2]:
path = r'C:\Users\hp\Documents\Analyse Bewerbungen'
all_files = glob.glob(os.path.join(path, '*.csv'))

df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=False)

In [3]:
df = df.reset_index().rename(columns={'index':'RowID'})

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1138 entries, 0 to 1137
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   RowID             1138 non-null   int64 
 1   suggestion        1138 non-null   object
 2   opinion           1138 non-null   object
 3   company           1138 non-null   object
 4   ausbildung place  1138 non-null   object
 5   ausbildung begin  1138 non-null   int64 
 6   salary            416 non-null    object
 7   study condition   1138 non-null   object
 8   beruf             1138 non-null   object
dtypes: int64(2), object(7)
memory usage: 80.1+ KB


In [5]:
df.shape

(1138, 9)

In [6]:
df.isna().sum()

RowID                 0
suggestion            0
opinion               0
company               0
ausbildung place      0
ausbildung begin      0
salary              722
study condition       0
beruf                 0
dtype: int64

In [7]:
def get_opinions(opinion):
    opinion_on_firma = 'Wie gefällt dir die Ausbildung bei deiner Firma?'
    opinion_on_beruf = 'Wie gefällt dir dein Ausbildungsberuf?'
    
    opinions = {
        opinion_on_firma : '',
        opinion_on_beruf : '',
    }

    if opinion_on_beruf and opinion_on_firma in opinion:
        get_firma_opinion = str(opinion).split('Wie gefällt dir die Ausbildung bei deiner Firma?')[1]
        get_firma_opinion = get_firma_opinion.split('Wie gefällt dir dein Ausbildungsberuf?')[0]
        get_beruf_opinion = str(opinion).split('Wie gefällt dir die Ausbildung bei deiner Firma?')[1]
        get_beruf_opinion = get_beruf_opinion.split('Wie gefällt dir dein Ausbildungsberuf?')[1]

        opinions[opinion_on_beruf] = get_beruf_opinion
        opinions[opinion_on_firma] = get_firma_opinion
    elif opinion_on_firma not in opinion:
        get_beruf_opinion = str(opinion).split('Wie gefällt dir die Ausbildung bei deiner Firma?')[0]
        get_beruf_opinion = get_beruf_opinion.split('Wie gefällt dir dein Ausbildungsberuf?')[1]

        opinions[opinion_on_beruf] = get_beruf_opinion

    return pd.Series([opinions[opinion_on_firma], opinions[opinion_on_beruf]],
            index=['firma opinion', 'beruf opinion'])

In [8]:
df[['firma opinion','beruf opinion']] = df['opinion'].apply(get_opinions)

In [9]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [10]:
def assign_salaries(salary):
    years_salaries = {1: None, 2: None, 3: None}

    if pd.isna(salary):
        return pd.Series([None, None, None],
                         index=['first year salary', 'second year salary', 'third year salary'])

    pattern = r'(\d)\.\s*Ausbildungsjahr\:(\d+)'
    matches = re.findall(pattern, str(salary))

    for year, sal in matches:
        year = int(year)
        if year in years_salaries:
            years_salaries[year] = int(sal)

    return pd.Series([years_salaries[1], years_salaries[2], years_salaries[3]],
                     index=['first year salary', 'second year salary', 'third year salary'])

In [11]:
df[['first year salary', 'second year salary', 'third year salary']] = df['salary'].apply(assign_salaries)

In [12]:
# Because there is a row that has 1€ or 0€ for salary
df['first year salary'] = df['first year salary'].replace(1, np.nan)
df['second year salary'] = df['second year salary'].replace(1, np.nan)
df['third year salary'] = df['third year salary'].replace(1, np.nan)

In [13]:
df['suggestion'] = df['suggestion'].replace({'Ich würde diese Firma weiterempfehlen!': 'positive', 'Ich würde diese Firma nicht weiterempfehlen!': 'negative', 'Unknown': None})

In [14]:
df['firma opinion'] = df['firma opinion'].replace('', None)

In [15]:
df = df.dropna(subset=['suggestion'])

In [16]:
from transformers import pipeline

sentiment_model = pipeline(
    'sentiment-analysis',
    model="oliverguhr/german-sentiment-bert"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [17]:
def get_sentiment_and_score(text):
    text = str(text)
    output = sentiment_model(text)[0]

    return pd.Series([output['label'], output['score']],
        index=['firma_sentiment_class', 'firma_sentiment_score'])

In [18]:
df[['firma_sentiment_class', 'firma_sentiment_score']] = df['firma opinion'].apply(get_sentiment_and_score)

In [19]:
df[['beruf_sentiment_class', 'beruf_sentiment_score']] = df['beruf opinion'].apply(get_sentiment_and_score)

In [20]:
df = df.drop(columns='RowID')

In [21]:
df.head()

,suggestion,opinion,company,ausbildung place,ausbildung begin,salary,study condition,beruf,firma opinion,beruf opinion,first year salary,second year salary,third year salary,firma_sentiment_class,firma_sentiment_score,beruf_sentiment_class,beruf_sentiment_score
0,positive,"Wie gefällt dir die Ausbildung bei deiner Firma?Nach etwa einem halben Jahr kann ich über die Abteilung der Ausbildung nur positive Worte finden. Ich finde besonders gut, dass ich in den Praxisphasen in unterschiedlichen Abteilungen eingesetzt werde und somit in unterschiedlichen Bereichen Berufserfahrung sammeln kann. Dadurch wird es mir vermutlich leichter fallen später zu entscheiden, in welchem Bereich ich genau arbeiten möchte. \r\n\r\nMeiner Meinung nach könnte der praktische Teil des dualen Studiums größer sein. Es gibt 3 Praxisphasen, die ca. 2-3 Monate gehen und ein sogenanntes Projektstudium im Unternehmen für zirka 7 Monate.Wie gefällt dir dein Ausbildungsberuf?Mein duales Studium gefällt mir sehr gut, da ich neben Theorie auch Praxiserfahrung sammeln kann. Besonders gut finde ich zudem, dass wir dadurch die Möglichkeit erhalten uns im Unternehmen zu vernetzen, da wir in verschiedenen Abteilungen arbeiten können und die Ausbildungsabteilung hohen Wert darauf legt, dass wir uns vernetzen.\r\n\r\nNegative Punkte habe ich bis jetzt noch nicht.",AUMOVIO Deutschland GmbH,Frankfurt am Main,2025,1. Ausbildungsjahr:1482€2. Ausbildungsjahr:1579€3. Ausbildungsjahr:1674€,Noch in der Ausbildung,duales studium it-secutity,"Nach etwa einem halben Jahr kann ich über die Abteilung der Ausbildung nur positive Worte finden. Ich finde besonders gut, dass ich in den Praxisphasen in unterschiedlichen Abteilungen eingesetzt werde und somit in unterschiedlichen Bereichen Berufserfahrung sammeln kann. Dadurch wird es mir vermutlich leichter fallen später zu entscheiden, in welchem Bereich ich genau arbeiten möchte. \r\n\r\nMeiner Meinung nach könnte der praktische Teil des dualen Studiums größer sein. Es gibt 3 Praxisphasen, die ca. 2-3 Monate gehen und ein sogenanntes Projektstudium im Unternehmen für zirka 7 Monate.","Mein duales Studium gefällt mir sehr gut, da ich neben Theorie auch Praxiserfahrung sammeln kann. Besonders gut finde ich zudem, dass wir dadurch die Möglichkeit erhalten uns im Unternehmen zu vernetzen, da wir in verschiedenen Abteilungen arbeiten können und die Ausbildungsabteilung hohen Wert darauf legt, dass wir uns vernetzen.\r\n\r\nNegative Punkte habe ich bis jetzt noch nicht.",1482.0,1579.0,1674.0,positive,0.991002,positive,0.994992
1,positive,Wie gefällt dir die Ausbildung bei deiner Firma?Wir wurden herzlich empfangen und eingearbeitet. Desweiteren wird man die gesamte Zeit über unterstützt und jederzeit fragen stellen.Wie gefällt dir dein Ausbildungsberuf?Praxisnahe und spannende Aufgaben mit guter Unterstützung. Möglichkeit viele unterschiedliche Abteilungen kennenzulernen und mit vielen Personen zu reden.,AUMOVIO Deutschland GmbH,Frankfurt am Main,2025,NaN,Noch in der Ausbildung,duales studium it-secutity,Wir wurden herzlich empfangen und eingearbeitet. Desweiteren wird man die gesamte Zeit über unterstützt und jederzeit fragen stellen.,Praxisnahe und spannende Aufgaben mit guter Unterstützung. Möglichkeit viele unterschiedliche Abteilungen kennenzulernen und mit vielen Personen zu reden.,NaN,NaN,NaN,positive,0.999710,positive,0.999593
2,positive,"Wie gefällt dir die Ausbildung bei deiner Firma?Es gibt viele gute Vergütungen als Duali. Es wird sich gut um die Dualis gekümmert. Bessere wäre es, wenn es noch open co-work spaces, Büros für jeder Mann und nicht nur für Abteilungen geben würde.Wie gefällt dir dein Ausbildungsberuf?Mir gefallen die Berufsaussichten, nach dem Dualen Studium. Mir fehlt es aber noch an thematischer Tiefe im Studium.",AUMOVIO Deutschland GmbH,Frankfurt am Main,2024,NaN,Noch in der Ausbildung,duales studium it-secutity,"Es gibt viele gute Vergütungen als Duali. Es wird sich gut um die Dualis gekümmert. Bessere wäre es, 

In [22]:
def combine_sentiments(df):
    threshold = 0.5
    col1, score1 = df['firma_sentiment_class'], df['firma_sentiment_score']
    col2, score2 = df['beruf_sentiment_class'], df['beruf_sentiment_score']
    labels = {col1, col2}

    if labels == {'positive'}:
        return 'positive'
    if labels == {'negative'}:
        return 'negative'
    if labels == {'positive', 'neutral'}:
        return 'positive'
    if labels == {'negative', 'neutral'}:
        return 'negative'
    if labels == {'neutral'}:
        final_score = (score1 + score2) / 2
        return 'positive' if final_score >= threshold else 'negative'
    if labels == {'positive', 'negative'}:
        score1 = (score1 if col1 == 'positive' else -score1)
        score2 = (score2 if col2 == 'positive' else -score2)
        return 'positive' if (score1 + score2) >= 0 else 'negative'

In [23]:
df['combined_sentiment'] = df.apply(combine_sentiments, axis=1)

In [24]:
df['combined_sentiment'].value_counts()

combined_sentiment
positive    1068
negative      59
Name: count, dtype: int64

In [25]:
df['suggestion'].value_counts()

suggestion
positive    1095
negative      32
Name: count, dtype: int64

In [26]:
f1_s = f1_score(df['suggestion'], df['combined_sentiment'], pos_label='positive', average='binary')
print('F1-score:', f1_s)
print('\nFull report:', classification_report(df['suggestion'], df['combined_sentiment']))
print('\nConfusion Matrix:', confusion_matrix(df['suggestion'], df['combined_sentiment']))

F1-score: 0.9690245030050856

Full report:               precision    recall  f1-score   support

    negative       0.20      0.38      0.26        32
    positive       0.98      0.96      0.97      1095

    accuracy                           0.94      1127
   macro avg       0.59      0.67      0.62      1127
weighted avg       0.96      0.94      0.95      1127


Confusion Matrix: [[  12   20]
 [  47 1048]]


In [27]:
df['combined opinions'] = "Firma: " + df['firma opinion'].astype(str) + " Beruf: " + df['beruf opinion'].astype(str)

In [28]:
label2id = {
    'negative': 0,
    'positive': 1}
id2label = {
    1: 'positive',
    0: 'negative'}
df['label'] = df['suggestion'].map(label2id)

In [29]:
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)
print('Train size:', len(train_df), 'Validation size:', len(val_df), 'Test size:', len(test_df))

Train size: 788 Validation size: 169 Test size: 170


In [30]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [31]:
base_model = 'oliverguhr/german-sentiment-bert'
tokenizer = AutoTokenizer.from_pretrained(base_model)

def tokenize(batch):
    # padding=True means feed each text in a batch with empty tokens or 0 the texts in a batch must have the same length
    # truncation=True means to cut the rest of tokens if a text has more than 512 token
    # the model treat the texts as a batch(couple of texts in one group)
    return tokenizer(batch['combined opinions'], padding=True, truncation=True, max_length=512)

In [32]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[['combined opinions', 'label']]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(val_df[['combined opinions', 'label']]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df[['combined opinions', 'label']]).map(tokenize, batched=True)

Map:   0%|          | 0/788 [00:00<?, ? examples/s]

Map:   0%|          | 0/169 [00:00<?, ? examples/s]

Map:   0%|          | 0/170 [00:00<?, ? examples/s]

In [33]:
model = AutoModelForSequenceClassification.from_pretrained(
    base_model,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True # ign.. makes the model to not show an error in case of converting the output classes from 2 to 3 or something like this
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: oliverguhr/german-sentiment-bert
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [34]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    
    return {'f1': f1_score(labels, predictions)}

In [35]:
training_args = TrainingArguments(
    output_dir='./finetuned_sentiment',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1'
)

In [37]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)
trainer.train(resume_from_checkpoint=True)
results = trainer.evaluate(test_ds)

Epoch,Training Loss,Validation Loss


C:\Users\hp\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Training Loss,Validation Loss,Epoch,F1
No log,0.123663,2,0.990991


In [38]:
print('F1-score after fine-tuning:', results['eval_f1'])

F1-score after fine-tuning: 0.990990990990991


In [39]:
import torch

In [43]:
def predict_sentiment(text, model, tokenizer, id2label):
    inputs = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors='pt')
    # Tells pyTorch that we aren't goint to not prepare for a training part that waste memory
    with torch.no_grad(): 
        # always unpack the tokenizer output when using the model, because it needs separated arguments
        outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1).numpy()[0]
    predicted_id = int(probs.argmax())

    return id2label[predicted_id], float(probs[predicted_id])

def predict_row(row):
    label, score = predict_sentiment(row['combined opinions'], model, tokenizer, id2label)

    return pd.Series([label, score], 
        index=['finetuned sentiment', 'finetuned score'])

In [44]:
df[['finetuned sentiment', 'finetuned score']] = df.apply(predict_row, axis=1)

In [46]:
df['finetuned sentiment'].value_counts()

finetuned sentiment
positive    1104
negative      23
Name: count, dtype: int64

In [48]:
df.to_csv('Azubis_Bewertungen.csv')

In [36]:
import pandas as pd
import numpy as np

In [45]:
data = pd.read_csv('analys_result.csv', index_col=0)
data[['first year salary', 'second year salary', 'third year salary']] = data[['first year salary', 'second year salary', 'third year salary']].replace(0, np.nan)
data.rename(columns={'suggestion':'sentiment'}, inplace=True)
data.to_csv('analys_result.csv', index=0)

In [46]:
data['study condition'].value_counts()

study condition
Noch in der Ausbildung            736
Übernommen                        365
Nicht Übernommen                   14
Wollte nicht übernommen werden     12
Name: count, dtype: int64